# PBO Analysis — Probability of Backtest Overfitting

Implements CSCV (Combinatorially Symmetric Cross-Validation) from
Bailey, Borwein, Lopez de Prado & Zhu (2017) to validate that our
SqueezeBreakout v7 optimization and blender weight optimization are
not overfit.

**Methodology:** Split returns into S sub-samples, exhaustively combine
C(S, S/2) train/test splits, measure how often the IS-optimal config
ranks below median OOS.  PBO < 0.50 → GO for deployment.

In [ ]:
# ── Cell 1: Imports and Setup ──────────────────────────────────────
import sys, types
sys.path.insert(0, '../src')

# Create 'app' namespace alias
app = types.ModuleType('app')
app.__path__ = ['../src/libs', '../src/apps']
sys.modules['app'] = app

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
import math
import time
import warnings
from datetime import datetime, timezone
from scipy import stats
from tqdm import tqdm

from binance.um_futures import UMFutures

# Indicator batch functions
from libs.features.indicators.volatility.bollinger import _compute_bb_batch
from libs.features.indicators.volatility.keltner import _compute_keltner_batch
from libs.features.indicators.volatility.atr import _compute_atr_batch
from libs.features.indicators.trend.kama import _compute_kama_batch
from libs.features.indicators.trend.ema import _compute_ema_batch
from libs.features.indicators.momentum.adx import _compute_adx_batch
from libs.features.indicators.momentum.cci import _compute_cci_batch
from libs.features.indicators.momentum.mfi import _compute_mfi_batch
from libs.features.indicators.momentum.momentum import _compute_momentum_batch
from libs.features.indicators.momentum.macd import _compute_macd_batch
from libs.features.indicators.volume.ad_line import _compute_ad_line_batch

# Model & scoring
from libs.models.squeeze_breakout.model import SqueezeBreakoutModel
from libs.optim_utils.scoring import backtest_multi_tp, compute_sharpe, BARS_PER_YEAR

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
client = UMFutures()

print('Setup complete')

In [ ]:
# ── Cell 2: Data Fetching ─────────────────────────────────────────
_ALL_COLS = [
    'timestamp', 'open', 'high', 'low', 'close', 'volume', 'close_time',
    'quote_volume', 'trades', 'taker_buy_base', 'taker_buy_quote', 'ignore'
]

def fetch_ohlcv(symbol: str, interval: str, start_date: str, end_date: str) -> pd.DataFrame:
    start_ms = int(datetime.strptime(start_date, '%Y-%m-%d').replace(tzinfo=timezone.utc).timestamp() * 1000)
    end_ms = int(datetime.strptime(end_date, '%Y-%m-%d').replace(tzinfo=timezone.utc).timestamp() * 1000)
    all_rows = []
    cursor = start_ms
    while cursor < end_ms:
        raw = client.klines(symbol, interval, startTime=cursor, endTime=end_ms, limit=1500)
        if not raw:
            break
        all_rows.extend(raw)
        cursor = int(raw[-1][6]) + 1
        if len(raw) < 1500:
            break
        time.sleep(0.15)
    df = pd.DataFrame(all_rows, columns=_ALL_COLS)
    for c in ['open','high','low','close','volume','taker_buy_base']:
        df[c] = df[c].astype(float)
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
    df = df.drop_duplicates('timestamp').sort_values('timestamp').reset_index(drop=True)
    print(f'{symbol} {interval}: {len(df)} bars [{df.timestamp.iloc[0]} → {df.timestamp.iloc[-1]}]')
    return df

SYMBOLS = ['BTCUSDT', 'XRPUSDT', 'SOLUSDT', 'BNBUSDT', 'DOGEUSDT']
START, END = '2024-06-01', '2026-06-01'
TIMEFRAME = '4h'
ANN_FACTOR = BARS_PER_YEAR.get(TIMEFRAME, 2190)  # 6*365 = 2190 for 4h

data = {}
for sym in SYMBOLS:
    data[sym] = fetch_ohlcv(sym, TIMEFRAME, START, END)

print(f'\nTotal: {sum(len(v) for v in data.values())} bars across {len(SYMBOLS)} assets')

In [ ]:
# ── Cell 3: CSCV Implementation ──────────────────────────────────

def cscv_pbo(returns_matrix: np.ndarray, S: int = 16) -> tuple:
    """
    Combinatorially Symmetric Cross-Validation for PBO.

    Args:
        returns_matrix: T x N matrix where each column is the return series
                       from one strategy/parameter configuration.
        S: Number of equal-sized subsamples to split data into (must be even).

    Returns:
        pbo: Probability of backtest overfitting (0 to 1).
        logits: Array of logit values from each CSCV split.

    Algorithm:
    1. Split T rows into S equal sub-samples.
    2. For each combination C(S, S/2):
       a. Training set J = selected S/2 sub-samples
       b. Test set J-bar = remaining S/2 sub-samples
       c. Compute performance metric (Sharpe) for each config on J
       d. Select best config n* on J
       e. Compute rank of n* on J-bar relative to other configs
       f. Compute relative rank: w = rank(n*) / N
       g. Compute logit: lambda = log(w / (1-w))
    3. PBO = fraction of logits that are <= 0
    """
    assert S % 2 == 0, 'S must be even'
    T, N = returns_matrix.shape
    assert N >= 2, 'Need at least 2 configurations'

    # Split into S equal sub-samples (trim tail rows if not evenly divisible)
    block_size = T // S
    assert block_size > 0, f'Not enough rows ({T}) for S={S} sub-samples'
    trimmed = returns_matrix[: block_size * S]
    blocks = trimmed.reshape(S, block_size, N)

    half = S // 2
    combos = list(itertools.combinations(range(S), half))

    # Cap combinations if C(S, S/2) is huge (e.g. S=16 → 12870, manageable)
    MAX_COMBOS = 15000
    if len(combos) > MAX_COMBOS:
        rng = np.random.default_rng(42)
        idx = rng.choice(len(combos), MAX_COMBOS, replace=False)
        combos = [combos[i] for i in sorted(idx)]

    logits = []
    for train_idx in combos:
        test_idx = tuple(i for i in range(S) if i not in train_idx)

        # Aggregate blocks into train/test returns
        train_returns = np.concatenate([blocks[i] for i in train_idx], axis=0)  # (half*block_size, N)
        test_returns = np.concatenate([blocks[i] for i in test_idx], axis=0)

        # Sharpe for each config on train
        train_sharpe = np.zeros(N)
        test_sharpe = np.zeros(N)
        for j in range(N):
            tr = train_returns[:, j]
            te = test_returns[:, j]
            std_tr = np.std(tr)
            std_te = np.std(te)
            train_sharpe[j] = (np.mean(tr) / std_tr * math.sqrt(ANN_FACTOR)) if std_tr > 1e-12 else 0.0
            test_sharpe[j] = (np.mean(te) / std_te * math.sqrt(ANN_FACTOR)) if std_te > 1e-12 else 0.0

        # Best config on train
        n_star = np.argmax(train_sharpe)

        # Rank of n_star on test (1 = worst, N = best)
        oos_perf = test_sharpe[n_star]
        rank = np.sum(test_sharpe <= oos_perf)  # rank from bottom

        # Relative rank
        w = rank / N
        # Clamp to avoid log(0)
        w = np.clip(w, 1e-6, 1.0 - 1e-6)
        logit = np.log(w / (1.0 - w))
        logits.append(logit)

    logits = np.array(logits)
    pbo = float(np.mean(logits <= 0))
    return pbo, logits


print(f'CSCV function defined.  C(16,8) = {math.comb(16,8)} combinations per asset.')

In [ ]:
# ── Cell 4: Parameter Grid Construction ───────────────────────────

param_grid = {
    'kama_fast_period': [3, 5, 10, 15, 20],
    'kama_slow_period': [25, 30, 35, 40, 45],
    'mom_period':       [15, 20, 25, 30],
    'squeeze_lookback': [1, 2, 3],
    'ss_threshold':     [1, 2, 3, 4, 5],
}

# Fixed defaults for SS-voter params (not part of v7 optimization)
FIXED_SS_PARAMS = {
    'cci_period': 5,
    'adx_period': 14,
    'adx_threshold': 18.0,
    'ad_sma_period': 21,
    'mfi_period': 14,
    'mfi_sma_period': 9,
    'mom_lr_period': 14,
    'mom_lr_mom_period': 10,
}

# Build all combinations
keys = list(param_grid.keys())
values = list(param_grid.values())
all_configs = []
for combo in itertools.product(*values):
    cfg = dict(zip(keys, combo))
    cfg.update(FIXED_SS_PARAMS)
    all_configs.append(cfg)

# Optimized v7 configs for reference (map to grid param names)
V7_OPTIMIZED = {
    'BTCUSDT': {'kama_fast_period': 15, 'kama_slow_period': 44, 'mom_period': 29, 'squeeze_lookback': 2, 'ss_threshold': 1},
    'XRPUSDT': {'kama_fast_period': 4,  'kama_slow_period': 28, 'mom_period': 30, 'squeeze_lookback': 3, 'ss_threshold': 5},
    'SOLUSDT': {'kama_fast_period': 5,  'kama_slow_period': 43, 'mom_period': 19, 'squeeze_lookback': 1, 'ss_threshold': 1},
    'BNBUSDT': {'kama_fast_period': 5,  'kama_slow_period': 43, 'mom_period': 20, 'squeeze_lookback': 1, 'ss_threshold': 3},
    'DOGEUSDT':{'kama_fast_period': 3,  'kama_slow_period': 29, 'mom_period': 16, 'squeeze_lookback': 1, 'ss_threshold': 4},
}

print(f'Parameter grid: {" x ".join(str(len(v)) for v in values)} = {len(all_configs)} configurations')
print(f'Fixed SS-voter params: {list(FIXED_SS_PARAMS.keys())}')

In [ ]:
# ── Cell 5: Generate Returns Matrix ───────────────────────────────

def build_feature_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build the feature DataFrame that SqueezeBreakoutModel.batch_evaluate() expects.
    Computes all indicators using the batch functions and assembles with correct column names.
    """
    fdf = df[['timestamp', 'open', 'high', 'low', 'close', 'volume']].copy()
    fdf = fdf.set_index('timestamp').sort_index()

    o = fdf['open'].values
    h = fdf['high'].values
    l = fdf['low'].values
    c = fdf['close'].values
    v = fdf['volume'].values

    # Bollinger Bands (20, 2.0) — standard for squeeze detection
    bb_mid, bb_upper, bb_lower = _compute_bb_batch(c, 20, 2.0)
    fdf['BollingerBands_upper'] = bb_upper
    fdf['BollingerBands_lower'] = bb_lower

    # Keltner Channel (20, 1.5, 10)
    kc_mid, kc_upper, kc_lower = _compute_keltner_batch(h, l, c, 20, 1.5, 10)
    fdf['KeltnerChannel_upper'] = kc_upper
    fdf['KeltnerChannel_lower'] = kc_lower

    # CCI (5, default for SS voters)
    fdf['CCI'] = _compute_cci_batch(h, l, c, 5)

    # ADX (14) — returns (adx, plus_di, minus_di)
    adx_arr, plus_di, minus_di = _compute_adx_batch(h, l, c, 14)
    fdf['ADX_adx'] = adx_arr
    fdf['ADX_plus_di'] = plus_di
    fdf['ADX_minus_di'] = minus_di

    # AD Line
    fdf['ADLine'] = _compute_ad_line_batch(h, l, c, v)

    # MFI (14)
    fdf['MFI'] = _compute_mfi_batch(h, l, c, v, 14)

    # Momentum (10, default period)
    fdf['Momentum'] = _compute_momentum_batch(c, 10)

    return fdf


def generate_returns_matrix(
    feature_df: pd.DataFrame,
    configs: list[dict],
    sym: str,
) -> np.ndarray:
    """
    For each param config, run SqueezeBreakout batch_evaluate + multi-TP backtest.
    Returns T x N matrix of per-bar equity returns.
    """
    close = feature_df['close'].values
    high = feature_df['high'].values
    low = feature_df['low'].values
    T = len(close)
    N = len(configs)

    returns_matrix = np.zeros((T, N))

    # KAMA values depend on kama_fast_period and kama_slow_period —
    # precompute for each unique (fast, slow) pair to avoid redundant work.
    kama_cache = {}
    unique_kama = set((cfg['kama_fast_period'], cfg['kama_slow_period']) for cfg in configs)
    for fast_p, slow_p in unique_kama:
        # _compute_kama_batch(data, period, fast_period, slow_period)
        # period = fast_p for the fast KAMA, period = slow_p for the slow
        # The KAMA indicator uses: period=lookback for efficiency ratio,
        # fast_period & slow_period for smoothing constants.
        # For dual-KAMA: fast KAMA = short-period KAMA, slow KAMA = long-period KAMA.
        kama_fast = _compute_kama_batch(close, fast_p, 2, fast_p)
        kama_slow = _compute_kama_batch(close, slow_p, 2, slow_p)
        kama_cache[(fast_p, slow_p)] = (kama_fast, kama_slow)

    for j, cfg in enumerate(tqdm(configs, desc=f'{sym} grid')):
        # Set KAMA columns for this config
        fast_p = cfg['kama_fast_period']
        slow_p = cfg['kama_slow_period']
        kf, ks = kama_cache[(fast_p, slow_p)]
        feature_df['KAMA_fast'] = kf
        feature_df['KAMA_slow'] = ks

        model = SqueezeBreakoutModel(cfg)
        try:
            directions = model.batch_evaluate(feature_df)
            eq_ret, _ = backtest_multi_tp(
                directions.values, high, low, close,
                tp_pcts=(0.015, 0.03, 0.05),
                tp_portions=(0.40, 0.30, 0.30),
                sl_pct=0.02,
                commission_bps=4.0,
                trail_to_breakeven=True,
            )
            returns_matrix[:, j] = eq_ret
        except Exception:
            # Config produced no valid signals — leave as zeros
            pass

    return returns_matrix


# Build feature DataFrames and returns matrices per asset
feature_dfs = {}
returns_matrices = {}

for sym in SYMBOLS:
    print(f'\n=== {sym} ===')
    fdf = build_feature_df(data[sym])
    feature_dfs[sym] = fdf
    rm = generate_returns_matrix(fdf, all_configs, sym)
    returns_matrices[sym] = rm
    nonzero = np.sum(np.any(rm != 0, axis=0))
    print(f'  Returns matrix: {rm.shape}  ({nonzero}/{rm.shape[1]} configs produced trades)')

In [ ]:
# ── Cell 6: Run PBO Analysis per Asset ────────────────────────────

S = 16  # standard sub-sample count
PBO_THRESHOLD = 0.50

pbo_results = {}

for sym in SYMBOLS:
    rm = returns_matrices[sym]

    # Filter out configs that produced zero trades (all-zero columns)
    active_mask = np.any(rm != 0, axis=0)
    rm_active = rm[:, active_mask]

    if rm_active.shape[1] < 2:
        print(f'{sym}: Only {rm_active.shape[1]} active configs — skipping PBO (need >= 2)')
        pbo_results[sym] = {'pbo': np.nan, 'logits': np.array([]), 'n_active': rm_active.shape[1]}
        continue

    pbo, logits = cscv_pbo(rm_active, S=S)
    mean_logit = float(np.mean(logits))

    # OOS rank of v7 optimized config (if it exists in grid)
    v7 = V7_OPTIMIZED.get(sym, {})
    v7_idx = None
    for i, cfg in enumerate(all_configs):
        if all(cfg.get(k) == v7.get(k) for k in param_grid.keys()):
            v7_idx = i
            break

    verdict = 'GO' if pbo < PBO_THRESHOLD else 'NO-GO'
    pbo_results[sym] = {
        'pbo': pbo,
        'logits': logits,
        'mean_logit': mean_logit,
        'n_active': int(np.sum(active_mask)),
        'v7_in_grid': v7_idx is not None,
        'verdict': verdict,
    }
    print(f'{sym}: PBO = {pbo:.4f}  mean_logit = {mean_logit:+.3f}  '
          f'active = {rm_active.shape[1]}  → {verdict}')

In [ ]:
# ── Cell 7: PBO for Blender Weights ───────────────────────────────

# Blender ensemble: generate returns for different weight combinations
# applied to per-model per-regime signals.

# Optimized blender groups (4 regime groups)
BLENDER_GROUPS = {
    'TREND_BULL':  {'momentum': 1.00},
    'TREND_BEAR':  {'momentum': 0.49, 'squeeze_breakout': 0.51},
    'RANGE':       {'momentum': 1.00},
    'CHOPPY':      {'momentum': 0.13, 'squeeze_breakout': 0.87},
    'TRANSITION':  {'momentum': 0.78, 'squeeze_breakout': 0.22},
}

# Generate a grid of blender weight configs by varying the momentum/squeeze
# blend ratio in each regime group.
blend_ratios = np.arange(0.0, 1.05, 0.1)  # 0.0, 0.1, ..., 1.0 (11 steps)

def make_blend_configs(ratios: np.ndarray) -> list[dict]:
    """Generate blend configs: each is a dict mapping regime_group → (mom_w, sq_w)."""
    # Vary ratio across all 5 groups independently would be 11^5 ~ 161K.
    # Instead: vary with a single shared ratio shift for tractability.
    configs = []
    # Strategy: for each of the 5 groups, shift the momentum weight by delta
    for delta in np.arange(-0.3, 0.35, 0.05):  # 13 shifts
        for scale in np.arange(0.5, 1.6, 0.1):  # 11 scales
            cfg = {}
            valid = True
            for group, weights in BLENDER_GROUPS.items():
                mom_w = weights.get('momentum', 0.0)
                sq_w = weights.get('squeeze_breakout', 0.0)
                new_mom = np.clip(mom_w * scale + delta, 0.0, 1.0)
                new_sq = 1.0 - new_mom
                cfg[group] = {'momentum': new_mom, 'squeeze_breakout': max(0.0, new_sq)}
            configs.append(cfg)
    return configs


def blender_returns(
    feature_df: pd.DataFrame,
    mom_cfg: dict,
    sq_cfg: dict,
    blend_cfg: dict,
) -> np.ndarray:
    """
    Compute blended returns for a single blend config.
    mom_cfg/sq_cfg: the fixed optimized params for momentum and squeeze models.
    blend_cfg: dict mapping regime_group → {momentum: w1, squeeze_breakout: w2}

    Simplified: since we don't have regime labels in this standalone notebook,
    we use a uniform blend across all bars (weighted average of model returns).
    This tests whether the weight allocation itself is overfit.
    """
    close = feature_df['close'].values
    high = feature_df['high'].values
    low = feature_df['low'].values
    T = len(close)

    # Compute squeeze returns with the optimized squeeze config
    sq_model = SqueezeBreakoutModel(sq_cfg)
    sq_dirs = sq_model.batch_evaluate(feature_df)
    sq_eq, _ = backtest_multi_tp(
        sq_dirs.values, high, low, close,
        tp_pcts=(0.015, 0.03, 0.05),
        tp_portions=(0.40, 0.30, 0.30),
        sl_pct=0.02, commission_bps=4.0,
    )

    # Momentum proxy: use simple momentum crossover returns
    # (we don't have a MomentumModel in this notebook, so approximate
    #  with a simple RSI + MACD strategy)
    macd_l, macd_s, macd_h = _compute_macd_batch(close, 12, 26, 9)
    from libs.features.indicators.momentum.rsi import _compute_rsi_batch
    rsi = _compute_rsi_batch(close, 14)

    mom_dirs = np.zeros(T, dtype=int)
    long_mask = (rsi > 70) & (macd_h > 0) & ~np.isnan(rsi) & ~np.isnan(macd_h)
    short_mask = (rsi < 34) & (macd_h < 0) & ~np.isnan(rsi) & ~np.isnan(macd_h)
    mom_dirs[long_mask] = 1
    mom_dirs[short_mask] = -1

    mom_eq, _ = backtest_multi_tp(
        mom_dirs, high, low, close,
        tp_pcts=(0.015, 0.03, 0.05),
        tp_portions=(0.40, 0.30, 0.30),
        sl_pct=0.02, commission_bps=4.0,
    )

    # Average blend weight across groups (uniform regime assumption)
    mom_w_avg = np.mean([g.get('momentum', 0.0) for g in blend_cfg.values()])
    sq_w_avg = np.mean([g.get('squeeze_breakout', 0.0) for g in blend_cfg.values()])
    total = mom_w_avg + sq_w_avg
    if total > 0:
        mom_w_avg /= total
        sq_w_avg /= total

    blended = mom_w_avg * mom_eq + sq_w_avg * sq_eq
    return blended


# Build blender returns matrix using BTC as representative asset
blend_configs = make_blend_configs(blend_ratios)
print(f'Blender weight configs: {len(blend_configs)}')

# Use BTC feature_df with the v7-optimized squeeze params
btc_fdf = feature_dfs['BTCUSDT'].copy()
btc_v7 = {**V7_OPTIMIZED['BTCUSDT'], **FIXED_SS_PARAMS}

# Set KAMA columns for the optimized config
btc_fdf['KAMA_fast'] = _compute_kama_batch(
    btc_fdf['close'].values, btc_v7['kama_fast_period'], 2, btc_v7['kama_fast_period'])
btc_fdf['KAMA_slow'] = _compute_kama_batch(
    btc_fdf['close'].values, btc_v7['kama_slow_period'], 2, btc_v7['kama_slow_period'])

T_blend = len(btc_fdf)
N_blend = len(blend_configs)
blend_rm = np.zeros((T_blend, N_blend))

for j, bcfg in enumerate(tqdm(blend_configs, desc='Blender grid')):
    try:
        blend_rm[:, j] = blender_returns(btc_fdf, {}, btc_v7, bcfg)
    except Exception:
        pass

# Run CSCV on blender
active_blend = np.any(blend_rm != 0, axis=0)
blend_rm_active = blend_rm[:, active_blend]

if blend_rm_active.shape[1] >= 2:
    blend_pbo, blend_logits = cscv_pbo(blend_rm_active, S=S)
    blend_verdict = 'GO' if blend_pbo < PBO_THRESHOLD else 'NO-GO'
    pbo_results['Blender'] = {
        'pbo': blend_pbo,
        'logits': blend_logits,
        'mean_logit': float(np.mean(blend_logits)),
        'n_active': int(np.sum(active_blend)),
        'verdict': blend_verdict,
    }
    print(f'Blender: PBO = {blend_pbo:.4f}  mean_logit = {np.mean(blend_logits):+.3f}  → {blend_verdict}')
else:
    print(f'Blender: Only {blend_rm_active.shape[1]} active configs — cannot compute PBO')
    pbo_results['Blender'] = {'pbo': np.nan, 'logits': np.array([]), 'n_active': 0, 'verdict': 'N/A'}

In [ ]:
# ── Cell 8: Visualization ─────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# --- Plot 1: PBO bar chart ---
ax = axes[0, 0]
labels = [k for k in pbo_results if not np.isnan(pbo_results[k]['pbo'])]
pbos = [pbo_results[k]['pbo'] for k in labels]
colors = ['#2ecc71' if p < PBO_THRESHOLD else '#e74c3c' for p in pbos]
bars = ax.bar(labels, pbos, color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(y=PBO_THRESHOLD, color='red', linestyle='--', linewidth=1.5, label=f'Threshold = {PBO_THRESHOLD}')
ax.set_ylabel('PBO')
ax.set_title('Probability of Backtest Overfitting')
ax.legend()
ax.set_ylim(0, 1)
for bar, val in zip(bars, pbos):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# --- Plot 2: Logit distributions ---
ax = axes[0, 1]
for k in labels:
    logits = pbo_results[k]['logits']
    if len(logits) > 0:
        ax.hist(logits, bins=40, alpha=0.5, label=k, density=True)
ax.axvline(x=0, color='red', linestyle='--', linewidth=1.5)
ax.set_xlabel('Logit (λ)')
ax.set_ylabel('Density')
ax.set_title('Logit Distributions (λ ≤ 0 ⟹ overfit)')
ax.legend(fontsize=7)

# --- Plot 3: Stochastic dominance (IS vs OOS Sharpe ranks) ---
ax = axes[1, 0]
for sym in SYMBOLS:
    rm = returns_matrices[sym]
    active = np.any(rm != 0, axis=0)
    rm_a = rm[:, active]
    if rm_a.shape[1] < 2:
        continue
    T, N = rm_a.shape
    block_size = T // S
    half_T = block_size * (S // 2)
    is_returns = rm_a[:half_T]
    oos_returns = rm_a[half_T:half_T * 2]

    is_sharpe = np.array([
        (np.mean(is_returns[:, j]) / np.std(is_returns[:, j]) * math.sqrt(ANN_FACTOR))
        if np.std(is_returns[:, j]) > 1e-12 else 0.0
        for j in range(N)
    ])
    oos_sharpe = np.array([
        (np.mean(oos_returns[:, j]) / np.std(oos_returns[:, j]) * math.sqrt(ANN_FACTOR))
        if np.std(oos_returns[:, j]) > 1e-12 else 0.0
        for j in range(N)
    ])

    sorted_is = np.sort(is_sharpe)
    sorted_oos = np.sort(oos_sharpe)
    cdf = np.linspace(0, 1, len(sorted_is))
    ax.plot(sorted_is, cdf, label=f'{sym} IS', linestyle='-', alpha=0.7)
    ax.plot(sorted_oos, cdf, label=f'{sym} OOS', linestyle='--', alpha=0.7)

ax.set_xlabel('Annualized Sharpe')
ax.set_ylabel('CDF')
ax.set_title('IS vs OOS Sharpe CDF (Stochastic Dominance)')
ax.legend(fontsize=6, ncol=2)

# --- Plot 4: Summary heatmap ---
ax = axes[1, 1]
heatmap_data = []
heatmap_labels = []
for k in labels:
    r = pbo_results[k]
    heatmap_labels.append(k)
    heatmap_data.append([
        r['pbo'],
        r.get('mean_logit', 0.0),
        r.get('n_active', 0) / max(len(all_configs), 1),  # fraction active
    ])

hm = np.array(heatmap_data)
im = ax.imshow(hm, cmap='RdYlGn_r', aspect='auto')
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['PBO', 'Mean Logit', 'Active Frac'])
ax.set_yticks(range(len(heatmap_labels)))
ax.set_yticklabels(heatmap_labels)
for i in range(hm.shape[0]):
    for j in range(hm.shape[1]):
        ax.text(j, i, f'{hm[i, j]:.2f}', ha='center', va='center', fontsize=9,
                color='white' if abs(hm[i, j]) > 0.5 else 'black')
ax.set_title('PBO Summary Heatmap')
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig('pbo_analysis_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plots saved to research/pbo_analysis_results.png')

In [ ]:
# ── Cell 9: Summary Table ─────────────────────────────────────────

print('\n' + '='*75)
print('PBO Analysis Summary — SqueezeBreakout v7 + Blender Optimization')
print('='*75)
print(f'{"Asset":<12} {"PBO":>8} {"Mean Logit":>12} {"Active Cfgs":>12} {"Verdict":>10}')
print('-'*75)

all_go = True
warnings_list = []

for k in list(SYMBOLS) + ['Blender']:
    r = pbo_results.get(k)
    if r is None:
        continue
    pbo = r['pbo']
    ml = r.get('mean_logit', np.nan)
    na = r.get('n_active', 0)
    verdict = r.get('verdict', 'N/A')

    if np.isnan(pbo):
        print(f'{k:<12} {"N/A":>8} {"N/A":>12} {na:>12} {"SKIP":>10}')
    else:
        print(f'{k:<12} {pbo:>8.4f} {ml:>+12.3f} {na:>12} {verdict:>10}')

    if verdict == 'NO-GO':
        all_go = False
    if not np.isnan(pbo) and pbo > 0.40:
        warnings_list.append(k)

print('-'*75)
print(f'\nOverall: {"✅ GO — deploy with confidence" if all_go else "❌ NO-GO — overfit risk detected"}')

if warnings_list:
    print(f'\n⚠️  Risk flags (PBO > 0.40): {", ".join(warnings_list)}')
    print('   Consider: reduce param count, expand training window, or use regularization.')

## Conclusions

### Interpretation Guide
- **PBO < 0.30**: Strong evidence the optimization is NOT overfit. Safe to deploy.
- **PBO 0.30–0.50**: Borderline. Proceed with caution, monitor OOS performance closely.
- **PBO > 0.50**: Likely overfit. The IS-optimal configuration performs below median OOS more than half the time.

### Risk Flags
- Any asset with PBO > 0.40 should be reviewed for:
  1. **Too many degrees of freedom** — reduce the parameter grid or fix more params
  2. **Insufficient data** — 2 years of 4h bars (~4380 bars) may not be enough for some assets
  3. **Regime sensitivity** — some assets may have different behaviour across regimes, inflating apparent overfitting

### Remediation
- For overfit assets: Fix more params to defaults (reduce from 5-param to 3-param search)
- For borderline: Extend data window to 3 years, or use walk-forward validation
- For the blender: Consider fewer regime groups (merge TREND_BULL and RANGE)

### Next Steps
1. If all GO: proceed to paper trading with v7 params
2. If any NO-GO: iterate on the flagged asset's param space before deployment
3. Re-run this notebook quarterly to validate ongoing OOS performance

## Remediation: Reduced 3-Param Re-Optimization for BTC & DOGE

BTC (PBO=0.72) and DOGE (PBO=0.56) failed the 5-param PBO test.
Strategy: fix `squeeze_lookback` and `ss_threshold` at defaults (1 and 3),
optimize only `kama_fast_period`, `kama_slow_period`, `mom_period` (3 params, 100 configs).
Fewer degrees of freedom → lower overfit risk.

In [ ]:
# ── Cell 10: Reduced Grid for BTC & DOGE ──────────────────────────
# Fix squeeze_lookback=1, ss_threshold=3 (defaults). Only optimize 3 params.

reduced_grid = {
    'kama_fast_period': [3, 5, 10, 15, 20],
    'kama_slow_period': [25, 30, 35, 40, 45],
    'mom_period':       [15, 20, 25, 30],
}

FIXED_REDUCED = {
    'squeeze_lookback': 1,
    'ss_threshold': 3,
    **FIXED_SS_PARAMS,
}

reduced_keys = list(reduced_grid.keys())
reduced_values = list(reduced_grid.values())
reduced_configs = []
for combo in itertools.product(*reduced_values):
    cfg = dict(zip(reduced_keys, combo))
    cfg.update(FIXED_REDUCED)
    reduced_configs.append(cfg)

print(f'Reduced grid: {" x ".join(str(len(v)) for v in reduced_values)} = {len(reduced_configs)} configs')
print(f'Fixed params: squeeze_lookback={FIXED_REDUCED["squeeze_lookback"]}, ss_threshold={FIXED_REDUCED["ss_threshold"]}')

# Generate returns matrices for BTC and DOGE only
OVERFIT_ASSETS = ['BTCUSDT', 'DOGEUSDT']
reduced_returns = {}

for sym in OVERFIT_ASSETS:
    print(f'\n=== {sym} (reduced grid) ===')
    fdf = feature_dfs[sym].copy()
    rm = generate_returns_matrix(fdf, reduced_configs, sym)
    reduced_returns[sym] = rm
    nonzero = np.sum(np.any(rm != 0, axis=0))
    print(f'  Returns matrix: {rm.shape}  ({nonzero}/{rm.shape[1]} configs produced trades)')

In [ ]:
# ── Cell 11: PBO on Reduced Grid ─────────────────────────────────

reduced_pbo = {}

for sym in OVERFIT_ASSETS:
    rm = reduced_returns[sym]
    active_mask = np.any(rm != 0, axis=0)
    rm_active = rm[:, active_mask]

    if rm_active.shape[1] < 2:
        print(f'{sym}: Only {rm_active.shape[1]} active configs — skipping')
        reduced_pbo[sym] = {'pbo': np.nan, 'verdict': 'SKIP'}
        continue

    pbo_val, logits_val = cscv_pbo(rm_active, S=S)
    mean_logit = float(np.mean(logits_val))

    # Find best full-sample config
    full_sharpes = np.array([
        (np.mean(rm_active[:, j]) / np.std(rm_active[:, j]) * math.sqrt(ANN_FACTOR))
        if np.std(rm_active[:, j]) > 1e-12 else 0.0
        for j in range(rm_active.shape[1])
    ])
    best_idx = np.argmax(full_sharpes)
    best_sharpe = full_sharpes[best_idx]

    # Map back to config
    active_indices = np.where(active_mask)[0]
    best_cfg_idx = active_indices[best_idx]
    best_cfg = reduced_configs[best_cfg_idx]

    verdict = 'GO' if pbo_val < PBO_THRESHOLD else 'NO-GO'
    reduced_pbo[sym] = {
        'pbo': pbo_val,
        'mean_logit': mean_logit,
        'best_sharpe': best_sharpe,
        'best_config': {k: best_cfg[k] for k in reduced_grid.keys()},
        'n_active': int(np.sum(active_mask)),
        'verdict': verdict,
    }
    print(f'{sym} (3-param): PBO = {pbo_val:.4f}  mean_logit = {mean_logit:+.3f}  '
          f'best_sharpe = {best_sharpe:.2f}  → {verdict}')
    print(f'  Best config: {reduced_pbo[sym]["best_config"]}')

# Compare with defaults
print(f'\n{"="*70}')
print('COMPARISON: 5-param (overfit) vs 3-param (reduced) vs Defaults')
print(f'{"="*70}')
print(f'{"Asset":<12} {"5p PBO":>8} {"3p PBO":>8} {"3p Sharpe":>10} {"3p Verdict":>10}')
print(f'{"-"*48}')
for sym in OVERFIT_ASSETS:
    old = pbo_results.get(sym, {})
    new = reduced_pbo.get(sym, {})
    print(f'{sym:<12} {old.get("pbo", 0):.4f}   {new.get("pbo", 0):.4f}   '
          f'{new.get("best_sharpe", 0):>+10.2f} {new.get("verdict", "N/A"):>10}')

# Default params for comparison
DEFAULT_CFG = {
    'kama_fast_period': 5, 'kama_slow_period': 30, 'mom_period': 20,
    'squeeze_lookback': 1, 'ss_threshold': 3,
}
print(f'\nDefault params: {DEFAULT_CFG}')
for sym in OVERFIT_ASSETS:
    new = reduced_pbo.get(sym, {})
    bc = new.get('best_config', {})
    if bc == {k: DEFAULT_CFG[k] for k in reduced_grid.keys()}:
        print(f'  {sym}: Best 3-param config IS the default — no improvement from optimization')
    else:
        print(f'  {sym}: Best 3-param config: {bc} — potential improvement over defaults')